# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to Azure AI Foundry Agent Service using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.

## Setup

Before running this notebook, make sure you have:

1. **An Azure AI Foundry project** with a deployed chat model (e.g. `gpt-4o-mini`).
2. **Logged in with the Azure CLI** — run `az login` in your terminal.
3. **Set the required environment variables:**
   - `AZURE_AI_PROJECT_ENDPOINT` — your Azure AI Foundry project endpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.

The cell below installs the Python packages you need.

In [1]:
%pip install agent-framework azure-ai-projects azure-identity -q

Note: you may need to restart the kernel to use updated packages.


In [44]:
import os
import json
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.ai.projects.models import PromptAgentDefinition, Tool, FunctionTool
from openai.types.responses.response_input_param import FunctionCallOutput

# 1. Initialize the Project Client
project_client = AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential()
)

openai = project_client.get_openai_client()

agent = project_client.agents.create_version(
    agent_name="Friendly",
    definition=PromptAgentDefinition(
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        instructions="You are a helpful assistant that answers general questions.",
    ),
)

print(f"Created agent: name={agent.name}, version={agent.version}")

Created agent: name=Friendly, version=1


## Creating Your First Agent

An agent needs two things:

- **Instructions** that tell it *who it is* and *how to behave* (a system prompt).
- **Tools** — Python functions decorated with `@tool` that the agent can call to retrieve information or perform actions.

Below we define a simple tool that returns a list of popular vacation destinations. The agent will use this tool when a user asks for travel recommendations.

In [27]:
#@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [48]:
destination_tool = FunctionTool(
    name="get_destinations",
    description="Get a list of popular vacation destinations.",
    parameters={
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False,
    },
    strict=True
)

tools: list[Tool] = [destination_tool]

agent = project_client.agents.create_version(
    agent_name="friendlyAgent",
    definition=PromptAgentDefinition(
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        instructions=(
            "You are a helpful travel agent. Help users find their perfect vacation "
            "destination based on their preferences. Use the get_destinations tool "
            "to see available destinations."
        ),
        tools=tools
    )
)

print(f"Created agent: name={agent.name}, version={agent.version}")

response = openai.responses.create(
    input="I'm looking for a warm beach destination. What do you recommend?",
    extra_body={
        "agent_reference": {
            "name": agent.name,
            "type": "agent_reference",
        }
    },
)

tool_outputs = []

# Handle any requested function calls
for item in response.output:
    if item.type == "function_call":
        if item.name == "get_destinations":
            args = json.loads(item.arguments)
            result = get_destinations()

            tool_outputs.append(
                FunctionCallOutput(
                    type="function_call_output",
                    call_id=item.call_id,
                    output=json.dumps({"result": result}),
                )
            )

# If the agent requested a tool, send the tool output back
if tool_outputs:
    final_response = openai.responses.create(
        input=tool_outputs,
        previous_response_id=response.id,
        extra_body={
            "agent_reference": {
                "name": agent.name,
                "type": "agent_reference",
            }
        },
    )
    print("Final agent response:")
    print(final_response.output_text)
else:
    print("Agent response:")
    print(response.output_text)

Created agent: name=friendlyAgent, version=3
Final agent response:
Here are some great warm beach destinations you might consider:

1. **Bali, Indonesia** - Known for its beautiful beaches, vibrant culture, and stunning natural landscapes.

2. **Sydney, Australia** - Offers iconic beaches like Bondi and Manly, along with a fantastic city vibe.

3. **Cape Town, South Africa** - Features beautiful beaches with the backdrop of Table Mountain.

4. **Rio de Janeiro, Brazil** - Famous for Copacabana and Ipanema beaches, along with a lively atmosphere.

If you want more details on any of these destinations, just let me know!


## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.

In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

In this lesson you learned how to:

- **Create a provider** that connects to Azure AI Foundry Agent Service via `AzureAIProjectAgentProvider`.
- **Define a tool** using the `@tool` decorator so the agent can call your Python functions.
- **Run the agent** with a user message and print its response.
- **Stream responses** for real-time output.

In the next lesson we will explore agentic frameworks in more depth and learn how to give agents more powerful tools and multi-step reasoning capabilities.